# 01 — Data Cleaning & Validation

Wejście: `data/raw/Hotels.csv` · Wyjście: `data/processed/hotels_clean.parquet` + `data/sample/hotels_sample.csv`

**TODO:**
- konwersja serial Excela -> daty (Checkin/Checkout)
- obsługa ujemnych `Net_Revenue` / `GST` (anulacje)
- typy, sanity checks, walidacja
- zapis czystego pliku

In [71]:
import pandas as pd
import matplotlib.pyplot as plt 
import numpy as np
from pathlib import Path

In [72]:
CWD = Path.cwd()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Hotels.csv"

In [73]:
df = pd.read_csv(DATA_PATH)
df.head()
# Checkin and checkout are intigers, but they should be datetime


,Booking_ID,Hotel_Name,Hotel_Category,City,State,Country,Checkin_Date,Checkout_Date,Month,Quarter,...,Net_Revenue,GST_Amount,Total_With_GST,Booking_Channel,Customer_Type,Payment_Mode,Occupancy_Status,Customer_Rating,Feedback_Status,Covid_Impact_Level
0,BK9835534,ITC Kakatiya,Luxury,Hyderabad,Telangana,India,44320,44324,May,Q2,...,89160,10699.20,99859.20,Walk-in,Regular,UPI,Checked-Out,4.6,Positive,High
1,BK7803377,ITC Royal Bengal,Luxury,Kolkata,West Bengal,India,44352,44356,June,Q2,...,86830,10419.60,97249.60,Corporate,VIP,Credit Card,Checked-In,4.6,Positive,High
2,BK3686278,ITC Grand Goa,Resort,Goa,Goa,India,44443,44445,September,Q3,...,29389,3526.68,32915.68,Booking.com,Corporate,Cash,Checked-In,4.2,Positive,Low
3,BK7514886,Fortune Select Trinity,Business,Bengaluru,Karnataka,India,44469,44470,September,Q3,...,24409,2929.08,27338.08,Website,Regular,Debit Card,Cancelled,3.8,Neutral,Low
4,BK6312815,ITC Narmada,Business,Ahmedabad,Gujarat,India,44461,44463,September,Q3,...,9940,1192.80,11132.80,Walk-in,New Guest,UPI,Checked-In,4.6,Negative,Low


In [74]:
df.info()
df.describe()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 28 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Booking_ID          300000 non-null  str    
 1   Hotel_Name          300000 non-null  str    
 2   Hotel_Category      300000 non-null  str    
 3   City                300000 non-null  str    
 4   State               300000 non-null  str    
 5   Country             300000 non-null  str    
 6   Checkin_Date        300000 non-null  int64  
 7   Checkout_Date       300000 non-null  int64  
 8   Month               300000 non-null  str    
 9   Quarter             300000 non-null  str    
 10  Year                300000 non-null  int64  
 11  Room_Type           300000 non-null  str    
 12  Guests              300000 non-null  int64  
 13  Nights_Stayed       300000 non-null  int64  
 14  Room_Rate           300000 non-null  int64  
 15  Extra_Charges       300000 non-null  int64  


Booking_ID            0
Hotel_Name            0
Hotel_Category        0
City                  0
State                 0
Country               0
Checkin_Date          0
Checkout_Date         0
Month                 0
Quarter               0
Year                  0
Room_Type             0
Guests                0
Nights_Stayed         0
Room_Rate             0
Extra_Charges         0
Discount              0
Gross_Revenue         0
Net_Revenue           0
GST_Amount            0
Total_With_GST        0
Booking_Channel       0
Customer_Type         0
Payment_Mode          0
Occupancy_Status      0
Customer_Rating       0
Feedback_Status       0
Covid_Impact_Level    0
dtype: int64

In [75]:
df['Checkin_Date_time'] = pd.to_datetime(df['Checkin_Date'], unit='D', origin='1899-12-30')
df['Checkout_Date_time'] = pd.to_datetime(df['Checkout_Date'], unit='D', origin='1899-12-30')
df.head()

,Booking_ID,Hotel_Name,Hotel_Category,City,State,Country,Checkin_Date,Checkout_Date,Month,Quarter,...,Total_With_GST,Booking_Channel,Customer_Type,Payment_Mode,Occupancy_Status,Customer_Rating,Feedback_Status,Covid_Impact_Level,Checkin_Date_time,Checkout_Date_time
0,BK9835534,ITC Kakatiya,Luxury,Hyderabad,Telangana,India,44320,44324,May,Q2,...,99859.20,Walk-in,Regular,UPI,Checked-Out,4.6,Positive,High,2021-05-04,2021-05-08
1,BK7803377,ITC Royal Bengal,Luxury,Kolkata,West Bengal,India,44352,44356,June,Q2,...,97249.60,Corporate,VIP,Credit Card,Checked-In,4.6,Positive,High,2021-06-05,2021-06-09
2,BK3686278,ITC Grand Goa,Resort,Goa,Goa,India,44443,44445,September,Q3,...,32915.68,Booking.com,Corporate,Cash,Checked-In,4.2,Positive,Low,2021-09-04,2021-09-06
3,BK7514886,Fortune Select Trinity,Business,Bengaluru,Karnataka,India,44469,44470,September,Q3,...,27338.08,Website,Regular,Debit Card,Cancelled,3.8,Neutral,Low,2021-09-30,2021-10-01
4,BK6312815,ITC Narmada,Business,Ahmedabad,Gujarat,India,44461,44463,September,Q3,...,11132.80,Walk-in,New Guest,UPI,Checked-In,4.6,Negative,Low,2021-09-22,2021-09-24


In [76]:
print(df['Checkin_Date_time'].max(), df['Checkin_Date_time'].min())
print(df['Checkout_Date_time'].max(), df['Checkout_Date_time'].min())
# Checkin and checkout dates are between 2021 and 2022, which is good.

2022-03-31 00:00:00 2021-04-01 00:00:00
2022-04-05 00:00:00 2021-04-02 00:00:00


In [77]:
df['nights_calc'] = (df['Checkout_Date_time'] - df['Checkin_Date_time']).dt.days
print(df.head())
print('Checkout < Checkin:', (df['Checkout_Date_time'] < df['Checkin_Date_time']).sum())
print('nights_calc != Nights_Stayed:', (df['nights_calc'] != df['Nights_Stayed']).sum())
# Nights_Stayed and nights_calc are the same, which is good. There are no cases where checkout is before checkin, which is also good.

  Booking_ID              Hotel_Name Hotel_Category       City        State  \
0  BK9835534            ITC Kakatiya         Luxury  Hyderabad    Telangana   
1  BK7803377        ITC Royal Bengal         Luxury    Kolkata  West Bengal   
2  BK3686278           ITC Grand Goa         Resort        Goa          Goa   
3  BK7514886  Fortune Select Trinity       Business  Bengaluru    Karnataka   
4  BK6312815             ITC Narmada       Business  Ahmedabad      Gujarat   

  Country  Checkin_Date  Checkout_Date      Month Quarter  ...  \
0   India         44320          44324        May      Q2  ...   
1   India         44352          44356       June      Q2  ...   
2   India         44443          44445  September      Q3  ...   
3   India         44469          44470  September      Q3  ...   
4   India         44461          44463  September      Q3  ...   

   Booking_Channel Customer_Type  Payment_Mode  Occupancy_Status  \
0          Walk-in       Regular           UPI       Checked

In [78]:
print('Year incorrect:', (df['Year'] != df['Checkin_Date_time'].dt.year).sum())
print('Month incorrect:',(df['Month'] != df['Checkin_Date_time'].dt.strftime('%B')).sum())
# Maintaining data consistency and accurate documentation 

Year incorrect: 0
Month incorrect: 0


In [79]:
print(df['Gross_Revenue'].describe())
print(df['Net_Revenue'].isnull().sum())
print(df['Net_Revenue'].describe())


count    300000.000000
mean      52216.514390
std       32220.392052
min        4581.000000
25%       26118.000000
50%       44941.500000
75%       73198.000000
max      146886.000000
Name: Gross_Revenue, dtype: float64
0
count    300000.000000
mean      49716.107807
std       32255.358866
min        -211.000000
25%       23599.000000
50%       42477.500000
75%       70699.000000
max      145752.000000
Name: Net_Revenue, dtype: float64


In [85]:
revenue_check = pd.crosstab(index=df['Occupancy_Status'], columns=df['Net_Revenue'] <= 0)
print(revenue_check)

Net_Revenue        False  True 
Occupancy_Status               
Cancelled          29698      0
Checked-In        150158      2
Checked-Out       105215      0
No Show            14927      0


In [86]:
revenue_check_net_rev = df[df['Net_Revenue']<= 0]
print(revenue_check_net_rev)

       Booking_ID     Hotel_Name Hotel_Category  City          State Country  \
63092   BK9465293     ITC Mughal         Resort  Agra  Uttar Pradesh   India   
128401  BK3355219  ITC Grand Goa         Resort   Goa            Goa   India   

        Checkin_Date  Checkout_Date     Month Quarter  ...  Booking_Channel  \
63092          44473          44474   October      Q4  ...       MakeMyTrip   
128401         44616          44617  February      Q1  ...          Website   

       Customer_Type  Payment_Mode  Occupancy_Status  Customer_Rating  \
63092         Member   Credit Card        Checked-In              4.6   
128401     Corporate   Net Banking        Checked-In              3.5   

        Feedback_Status  Covid_Impact_Level  Checkin_Date_time  \
63092           Neutral                 Low         2021-10-04   
128401         Positive                 Low         2022-02-24   

        Checkout_Date_time  nights_calc  
63092           2021-10-05            1  
128401          20

In [87]:
    Net_gst = pd.crosstab(index=df['Occupancy_Status'], columns=df['GST_Amount'] <= 0)
    print(Net_gst)
    print(df[df['GST_Amount'] <= 0])

GST_Amount         False  True 
Occupancy_Status               
Cancelled          29698      0
Checked-In        150158      2
Checked-Out       105215      0
No Show            14927      0
       Booking_ID     Hotel_Name Hotel_Category  City          State Country  \
63092   BK9465293     ITC Mughal         Resort  Agra  Uttar Pradesh   India   
128401  BK3355219  ITC Grand Goa         Resort   Goa            Goa   India   

        Checkin_Date  Checkout_Date     Month Quarter  ...  Booking_Channel  \
63092          44473          44474   October      Q4  ...       MakeMyTrip   
128401         44616          44617  February      Q1  ...          Website   

       Customer_Type  Payment_Mode  Occupancy_Status  Customer_Rating  \
63092         Member   Credit Card        Checked-In              4.6   
128401     Corporate   Net Banking        Checked-In              3.5   

        Feedback_Status  Covid_Impact_Level  Checkin_Date_time  \
63092           Neutral                 Low

In [92]:
Total_gst = pd.crosstab(index=df['Occupancy_Status'], columns=df['Total_With_GST'] <= 0)
print(Total_gst)
print(df[df['Total_With_GST'] <= 0])

Total_With_GST     False
Occupancy_Status        
Cancelled          29698
Checked-In        150158
Checked-Out       105215
No Show            14927
Empty DataFrame
Columns: [Booking_ID, Hotel_Name, Hotel_Category, City, State, Country, Checkin_Date, Checkout_Date, Month, Quarter, Year, Room_Type, Guests, Nights_Stayed, Room_Rate, Extra_Charges, Discount, Gross_Revenue, Net_Revenue, GST_Amount, Total_With_GST, Booking_Channel, Customer_Type, Payment_Mode, Occupancy_Status, Customer_Rating, Feedback_Status, Covid_Impact_Level, Checkin_Date_time, Checkout_Date_time, nights_calc]
Index: []

[0 rows x 31 columns]


# Delete 2 rows because the data indicates that 2 people are checked in and checked out at the same time

In [93]:
values_to_drop = ['BK9465293', 'BK3355219']
df = df[~df['Booking_ID'].isin(values_to_drop)]
print(df.shape)

(299998, 31)
